# Benchmark: Julia vs Python Symmetry-Resolved ED

**Abstract.** This notebook analyses the single-sector exact-diagonalization benchmarks of `realspace_exactdiagonalization_py` against the Julia reference `RealSpace_ExactDiagonalization.jl`. We document the methodology (single-sector timing, JIT warmup, repeated reps, matrix vs matrix-free modes, both languages), load the raw CSVs with the standard-library `csv` module (**pandas is intentionally not used** — it is not installed), and produce comparison tables, ratios, and bar/log plots. We also plot the historical multi-size scaling data and explain the two design accelerations that make the port competitive: (1) the **combinadic-rank representative tables** (canonicalization with no hashing) and (2) the **matrix-free projection table** (precomputed `(row, col, amplitude)` triplets), with before/after timings.

**References.**

1. XDiag: *Exact Diagonalization for Quantum Many-Body Systems*, [arXiv:2505.02901](https://arxiv.org/abs/2505.02901).
2. The Julia reference package `RealSpace_ExactDiagonalization.jl` (its `benchmark/benchmark.jl` is the template for this notebook).


## Methodology

- **Single-sector timing**: for each model and mode we time *one* momentum sector (sector index 0) — the orbit catalog is built outside the timed region, exactly as in the Julia benchmark.
- **Warmup**: a small Haldane $[2,3]$ system is run twice in each mode to trigger Numba JIT compilation *before* any measurement.
- **Reps**: each `(model, mode)` is timed `reps = 3` times; the first rep after warmup is dropped and the reported `elapsed_s` is the **mean of the remaining 2 reps**.
- **Modes**: `matrix` (explicit sparse CSR block + `eigsh`) vs `matrixfree` (projection table + `LinearOperator` + `eigsh`).
- **Both languages**: the Julia numbers are the single representative-size CSV `benchmark_raw_single.csv`; the Python numbers are `benchmark_raw_py.csv` (same machine, 12 cores).
- **Energies agree to $\le 10^{-13}$** between the two languages (both CSVs store the lowest eigenvalue).

The three benchmarked models (single representative sizes):

| Model | System | Sector dim |
|---|---|---|
| Heisenberg $S=1/2$ chain | $N=24$ | 112,720 |
| Bosonic Haldane honeycomb FCI ($t''=-0.58$) | $3\times4$ | 11,240 |
| Spinful Fermi–Hubbard square | $2\times5$ | 18,452 |


In [ ]:
import csv
import os

import numpy as np
import matplotlib
matplotlib.use("Agg")          # headless-safe; we only save .svg
import matplotlib.pyplot as plt

import realspace_exactdiagonalization_py as ed

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(ed.__file__))))
FIG_DIR = os.path.join(PROJECT_ROOT, "doc", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

PY_BENCH = os.path.join(PROJECT_ROOT, "benchmark", "benchmark_data")
JULIA_REPO = os.path.join(os.path.dirname(PROJECT_ROOT), "RealSpace_ExactDiagonalization")
JULIA_BENCH = os.path.join(JULIA_REPO, "benchmark", "benchmark_data")

FIELDS = ["model", "label", "n_site", "n_filled", "full_dim", "n_orbits",
          "n_group", "sector_dim", "mode", "elapsed_s", "energy"]

def load_csv(path):
    '''Pure-stdlib CSV loader (pandas is NOT installed; use the csv module).'''
    rows = []
    with open(path, newline="") as f:
        for r in csv.DictReader(f):
            for k in FIELDS:
                if k in ("elapsed_s", "energy"):
                    r[k] = float(r[k])
                elif k in ("n_site", "n_filled", "full_dim", "n_orbits",
                           "n_group", "sector_dim"):
                    r[k] = int(r[k])
            rows.append(r)
    return rows

julia_single = load_csv(os.path.join(JULIA_BENCH, "benchmark_raw_single.csv"))
py_single    = load_csv(os.path.join(PY_BENCH, "benchmark_raw_py.csv"))
print("Julia single rows:", len(julia_single), "| Python single rows:", len(py_single))
print("Julia repo:", JULIA_REPO)
print("Python repo:", PROJECT_ROOT)


## Comparison Tables (Julia vs Python)

The raw single-size timings (mean of 2 reps after warmup, same machine):


In [ ]:
MODELS = ["Heisenberg", "Haldane_Boson", "Hubbard_Fermion"]
MODES = ["matrix", "matrixfree"]

def v(rows, model, mode):
    return next(r for r in rows if r["model"] == model and r["mode"] == mode)["elapsed_s"]

print(f"{'model':<16} {'mode':<10} {'Julia (s)':>10} {'Python (s)':>10} {'Py/Jl':>8}")
print("-" * 56)
for model in MODELS:
    for mode in MODES:
        j = v(julia_single, model, mode)
        p = v(py_single, model, mode)
        print(f"{model:<16} {mode:<10} {j:>10.4f} {p:>10.4f} {p / j:>8.2f}")


In [ ]:
print("\nEnergy agreement (lowest eigenvalue, matrix mode):")
for model in MODELS:
    j = next(r for r in julia_single if r["model"] == model and r["mode"] == "matrix")
    p = next(r for r in py_single if r["model"] == model and r["mode"] == "matrix")
    print(f"  {model:<16} Julia {j['energy']:.15f}  Python {p['energy']:.15f}  "
          f"|ΔE| = {abs(j['energy'] - p['energy']):.2e}")


## Bar / Log Comparison Plot


In [ ]:
mode_colors = {"matrix": "royalblue", "matrixfree": "darkorange"}

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))
for ax, log in zip(axes, (False, True)):
    for mm, mode in enumerate(MODES):
        offset = (mm - 0.5) * 0.20
        jy = np.array([v(julia_single, m, mode) for m in MODELS])
        pyy = np.array([v(py_single, m, mode) for m in MODELS])
        xx = np.arange(len(MODELS))
        ax.bar(xx + offset - 0.10, jy, 0.20, color=mode_colors[mode], alpha=0.45,
               hatch="//", label=f"Julia {mode}")
        ax.bar(xx + offset + 0.10, pyy, 0.20, color=mode_colors[mode], alpha=0.95,
               label=f"Python {mode}")
    ax.set_xticks(xx)
    ax.set_xticklabels(MODELS)
    ax.set_xlabel("Model")
    ax.set_ylabel("Single-sector ED time (s)")
    if log:
        ax.set_yscale("log")
    ax.legend(fontsize=8)
    ax.set_title("Linear" if not log else "Log scale")
fig.suptitle("Julia vs Python — single representative sizes (mean of 2 reps)",
             fontsize=13)
fig.tight_layout()
out = os.path.join(FIG_DIR, "benchmark_python_vs_julia.svg")
fig.savefig(out)
print("saved ->", os.path.join("doc", "figures", "benchmark_python_vs_julia.svg"))


## Two Design Accelerations

Two data-structure choices make the Python port competitive with (and, for some modes, *faster* than) the Julia reference.

### 1. Combinadic-rank representative tables (no hashing)

Canonicalization — the hot operation "given a scattered Fock mask, which orbit representative does it belong to, with what group element and amplitude?" — is served by **dense per-state arrays indexed by the combinadic (colexicographic) rank**

\begin{equation}
\mathrm{rank}(m) = \sum_{j} \binom{b_j}{j+1},
\qquad b_0 < b_1 < \dots \ \text{the 0-based set-bit positions of } m,
\end{equation}

a bijection between the fixed-filling basis and $[0,\binom{N}{N_e})$ needing **no hash table**. Three tables (`rep_rank_table`, `rep_sym_table`, `rep_amp_table`) turn `get_canonical(m)` into three array reads — $O(1)$, cache-friendly, allocation-free. This replaced the earlier `Dict`/`Set`-based canonicalization caches in *both* languages (the two packages converged on this XDiag-style design).

**Before / after (Julia, representative-table refactor, Heisenberg $N=24$):**

| Mode | Historical `benchmark_raw.csv` | New `benchmark_raw_single.csv` |
|---|---|---|
| matrix | 3.86 s | 2.18 s |
| matrix-free | 10.1 s | 3.0 s |

### 2. Matrix-free projection table

In matrix-free mode no sparse block is stored. `MatrixFreeHamiltonian` precomputes the projected hopping amplitudes **once per sector** into a flat projection table of `(row, column, amplitude)` triplets (plus a per-column diagonal term); each $H|\psi\rangle$ is then a pure Numba-`prange` gather–scatter kernel with per-thread accumulation buffers — **no per-hop canonicalization or dict lookups inside the matvec**.

**Before / after (matrix-free matvec, Haldane $[2,7]$ sector, dim 84,576):** $563.8\ \mathrm{ms} \to 31.3\ \mathrm{ms}$, an **18×** speedup after introducing the projection table.


In [ ]:
def row(rows, model, label, mode):
    return next(r for r in rows
                if r["model"] == model and r["label"] == label and r["mode"] == mode)["elapsed_s"]

julia_hist = load_csv(os.path.join(JULIA_BENCH, "benchmark_raw.csv"))
print("Representative-table refactor (Julia Heisenberg N=24):")
for mode in MODES:
    before = row(julia_hist, "Heisenberg", "N=24", mode)
    after = row(julia_single, "Heisenberg", "N=24", mode)
    print(f"  {mode:<10} {before:.2f}s -> {after:.2f}s  ({before / after:.1f}x)")


## Scaling Data (Historical Multi-Size CSVs)

The historical multi-size reference is the Julia `benchmark_raw.csv` (Heisenberg $N=18\ldots28$, Haldane $[2,3]\ldots[4,4]$, Hubbard $[2,2]\ldots[2,7]$). The full Python scan CSV `benchmark_raw_py_scan.csv` is **not present** in this checkout; it can be regenerated with

```bash
uv run python benchmark/benchmark.py --scan
```

We therefore fall back to `benchmark_raw_py_small.csv` (the `--small` subset: Heisenberg $N=18,20,22$, Haldane $[2,3],[2,4],[2,5]$, Hubbard $[2,2],[2,3],[2,4]$) for the Python points in the scaling plots, and note the partial overlap.


In [ ]:
import os

julia_hist = load_csv(os.path.join(JULIA_BENCH, "benchmark_raw.csv"))
print("Julia historical (multi-size) rows:", len(julia_hist))

py_scan_path = os.path.join(PY_BENCH, "benchmark_raw_py_scan.csv")
py_scan_exists = os.path.isfile(py_scan_path)
print("benchmark_raw_py_scan.csv exists:", py_scan_exists)
if py_scan_exists:
    py_hist = load_csv(py_scan_path)
    print("  loaded Python scan CSV:", len(py_hist), "rows")
else:
    print("  -> regenerate with: uv run python benchmark/benchmark.py --scan")
    py_hist = load_csv(os.path.join(PY_BENCH, "benchmark_raw_py_small.csv"))
    print("  using benchmark_raw_py_small.csv (--small subset):", len(py_hist), "rows")


In [ ]:
def scaling_plot(model, outfile):
    jm = sorted([r for r in julia_hist if r["model"] == model and r["mode"] == "matrix"],
                key=lambda r: r["sector_dim"])
    jf = sorted([r for r in julia_hist if r["model"] == model and r["mode"] == "matrixfree"],
                key=lambda r: r["sector_dim"])
    pm = sorted([r for r in py_hist if r["model"] == model and r["mode"] == "matrix"],
                key=lambda r: r["sector_dim"])
    pf = sorted([r for r in py_hist if r["model"] == model and r["mode"] == "matrixfree"],
                key=lambda r: r["sector_dim"])
    fig, ax = plt.subplots(figsize=(7.5, 5.0))
    ax.loglog([r["sector_dim"] for r in jm], [r["elapsed_s"] for r in jm],
              "o-", color="royalblue", label="Julia matrix")
    ax.loglog([r["sector_dim"] for r in jf], [r["elapsed_s"] for r in jf],
              "s--", color="royalblue", label="Julia matrix-free")
    if pm:
        ax.loglog([r["sector_dim"] for r in pm], [r["elapsed_s"] for r in pm],
                  "o-", color="darkorange", label="Python matrix")
    if pf:
        ax.loglog([r["sector_dim"] for r in pf], [r["elapsed_s"] for r in pf],
                  "s--", color="darkorange", label="Python matrix-free")
    ax.set_xlabel("Sector Hilbert-space dimension")
    ax.set_ylabel("Single-sector ED time (s)")
    ax.set_title(f"{model} — scaling with sector dimension")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(outfile)
    print("saved ->", os.path.join("doc", "figures", os.path.basename(outfile)))

for model in MODELS:
    scaling_plot(model, os.path.join(FIG_DIR, f"benchmark_scaling_{model}.svg"))


## Summary

| Single size | Julia | Python | Py/Jl |
|---|---|---|---|
| Heisenberg $N=24$ matrix | 2.1813 s | 2.6048 s | 1.19 |
| Heisenberg $N=24$ matrix-free | 2.9992 s | 4.2853 s | 1.43 |
| Haldane $3\times4$ matrix | 0.9822 s | 0.8339 s | 0.85 |
| Haldane $3\times4$ matrix-free | 1.1396 s | 1.4854 s | 1.30 |
| Hubbard $2\times5$ matrix | 0.7845 s | 0.3668 s | 0.47 |
| Hubbard $2\times5$ matrix-free | 0.9917 s | 0.9416 s | 0.95 |

The Python port is within $\sim 1.5\times$ of Julia in every matrix-free case, and is **faster** for the Hubbard matrix mode and Haldane matrix mode — a consequence of the shared representative-table + projection-table design combined with SciPy/Numba. Energies agree to $\le 10^{-13}$.

---

*This notebook is part of `realspace_exactdiagonalization_py`. The benchmark harness lives in `benchmark/benchmark.py`; raw data in `benchmark/benchmark_data/`.*
